# BB + ATR Baseline Strategy

This notebook evaluates the **Bollinger Bands + ATR** baseline strategy
on BTC/USDT and ETH/USDT using historical OHLCV data.

The core strategy logic is implemented in:
`strategies/baselines/bb_atr.py`

This notebook focuses on:
- Strategy evaluation
- Cumulative returns
- Drawdown analysi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from strategies.baseline.bb_atr import (
    compute_indicators,
    generate_signals,
    deduplicate_signals,
)

In [ ]:
def evaluate_strategy(df):
    df = df.copy()

    df["returns"] = df["close"].pct_change()
    df["strategy_returns"] = df["signal"].shift(1) * df["returns"]

    df["cumulative_market_returns"] = (1 + df["returns"]).cumprod()
    df["cumulative_strategy_returns"] = (1 + df["strategy_returns"]).cumprod()

    df["drawdown"] = (
        df["cumulative_strategy_returns"]
        / df["cumulative_strategy_returns"].cummax()
        - 1
    )

    return df


def plot_results(df, title):
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    ax1.plot(df.index, df["strategy_returns"], label="Strategy Returns")
    ax1.set_title(f"{title} – Strategy Returns")
    ax1.legend()

    ax2.plot(df.index, df["cumulative_market_returns"], label="Market")
    ax2.plot(df.index, df["cumulative_strategy_returns"], label="Strategy")
    ax2.set_title(f"{title} – Cumulative Returns")
    ax2.legend()

    ax3.plot(df.index, df["drawdown"], color="red")
    ax3.set_title(f"{title} – Drawdown")

    plt.tight_layout()
    plt.show()

In [ ]:
# Load BTC data
btc = pd.read_csv("data/processed/btc_1d.csv")
btc["datetime"] = pd.to_datetime(btc["datetime"])
btc.set_index("datetime", inplace=True)

# Apply BB + ATR strategy
btc = compute_indicators(btc)
btc = generate_signals(btc)
btc = deduplicate_signals(btc)

# Evaluate
btc = evaluate_strategy(btc)

# Plot
plot_results(btc, "BTC/USDT – BB + ATR")

In [ ]:
# Load ETH data
eth = pd.read_csv("data/processed/eth_1d.csv")
eth["datetime"] = pd.to_datetime(eth["datetime"])
eth.set_index("datetime", inplace=True)

# Apply BB + ATR strategy
eth = compute_indicators(eth)
eth = generate_signals(eth)
eth = deduplicate_signals(eth)

# Evaluate
eth = evaluate_strategy(eth)

# Plot
plot_results(eth, "ETH/USDT – BB + ATR")

In [ ]:
import os

def save_experiment_summary(
    df,
    strategy_name,
    asset,
    timeframe,
    out_dir="results/csv_outputs"
):
    """
    Save a one-row CSV summarizing strategy performance.
    """

    os.makedirs(out_dir, exist_ok=True)

    total_return = df["cumulative_strategy_returns"].iloc[-1] - 1
    benchmark_return = df["cumulative_market_returns"].iloc[-1] - 1
    max_drawdown = df["drawdown"].min()
    win_rate = (df["strategy_returns"] > 0).mean()
    total_trades = df["signal"].diff().abs().sum() / 2

    row = {
        "strategy": strategy_name,
        "asset": asset,
        "timeframe": timeframe,
        "total_return_pct": round(total_return * 100, 2),
        "benchmark_return_pct": round(benchmark_return * 100, 2),
        "max_drawdown_pct": round(max_drawdown * 100, 2),
        "win_rate_pct": round(win_rate * 100, 2),
        "total_trades": int(total_trades),
    }

    df_out = pd.DataFrame([row])

    out_file = os.path.join(out_dir, f"{strategy_name}.csv")

    if os.path.exists(out_file):
        df_out.to_csv(out_file, mode="a", header=False, index=False)
    else:
        df_out.to_csv(out_file, index=False)

    print(f"✅ Saved summary → {out_file}")


In [ ]:
save_experiment_summary(
    df=btc,
    strategy_name="bb_atr",
    asset="BTC",
    timeframe="1D"
)

In [ ]:
save_experiment_summary(
    df=eth,
    strategy_name="bb_atr",
    asset="ETH",
    timeframe="1D"
)